# nba_api exploration (task 1.2)

Goal: get familiar with `nba_api`'s endpoints before building the data
pipeline (`fetch.py`, task 1.3), and document what actually works, what
doesn't, and why — rate limiting, timeouts, blocking — so the pipeline is
designed around real constraints instead of assumptions.


In [1]:
import time

import pandas as pd
from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import commonplayerinfo, playercareerstats, leaguegamelog


## 1. Static data (bundled with the library, no network call)

`nba_api.stats.static` ships a hardcoded snapshot of all players/teams and
their IDs. This is instant and reliable — useful for resolving a player's
name to the `player_id` that every live endpoint needs.


In [2]:
lebron = players.find_players_by_full_name("LeBron James")
lakers = teams.find_teams_by_full_name("Lakers")
print(lebron)
print(lakers)


[{'id': 2544, 'full_name': 'LeBron James', 'first_name': 'LeBron', 'last_name': 'James', 'is_active': True}]
[{'id': 1610612747, 'full_name': 'Los Angeles Lakers', 'abbreviation': 'LAL', 'nickname': 'Lakers', 'city': 'Los Angeles', 'state': 'California', 'year_founded': 1948}]


## 2. Live endpoints (`stats.nba.com`)

The endpoints the project actually needs data from:

- `CommonPlayerInfo` — bio/profile data for the player profile page (1.6)
- `PlayerCareerStats` — season-by-season stats, needed for both the
  profile page and the comparison page (1.6/1.7)
- `LeagueGameLog` — per-game logs, a candidate source for more granular
  analysis in V2

These all call `stats.nba.com` directly (not a public/documented REST
API — `nba_api` reverse-engineers the endpoints NBA.com's own site uses).
Let's try one and measure what happens.


In [3]:
def try_endpoint(name, fn, timeout=15):
    start = time.time()
    try:
        result = fn(timeout=timeout)
        df = result.get_data_frames()[0]
        elapsed = time.time() - start
        print(f"{name}: OK in {elapsed:.1f}s, {len(df)} rows")
        return df
    except Exception as e:
        elapsed = time.time() - start
        print(f"{name}: FAILED after {elapsed:.1f}s -> {type(e).__name__}: {e}")
        return None


player_id = lebron[0]["id"]  # 2544
df_info = try_endpoint(
    "CommonPlayerInfo",
    lambda timeout: commonplayerinfo.CommonPlayerInfo(player_id=player_id, timeout=timeout),
)


CommonPlayerInfo: FAILED after 18.4s -> ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=15)


In [4]:
# nba_api's own docs recommend a short delay between calls to avoid
# tripping rate limits, even when a single call succeeds.
time.sleep(0.6)

df_career = try_endpoint(
    "PlayerCareerStats",
    lambda timeout: playercareerstats.PlayerCareerStats(player_id=player_id, timeout=timeout),
)

time.sleep(0.6)

df_gamelog = try_endpoint(
    "LeagueGameLog",
    lambda timeout: leaguegamelog.LeagueGameLog(season="2023-24", timeout=timeout),
)


PlayerCareerStats: FAILED after 15.4s -> ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=15)


LeagueGameLog: FAILED after 15.4s -> ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=15)


## Findings: rate limiting & blocking

All three live calls above timed out (`ReadTimeout`) rather than
returning data or an explicit error status.

**Update (follow-up test, after this notebook was first executed):**
this notebook was originally run inside a cloud sandbox, and the first
version of this section attributed the timeouts to that sandbox's
datacenter IP being blocked by `stats.nba.com`'s bot protection. That
hypothesis was tested and disproved: the same `CommonPlayerInfo` call
was re-run from a home network (a residential IP, not a cloud IP) and
produced the *identical* `ReadTimeout` after the same ~15-30s window.
The static endpoints, re-tested the same way, again returned instantly.
So the failure is not IP-origin-specific.

The corrected explanation, grounded in `nba_api`'s own issue tracker
(this is a long-running, widely reported pattern, not a one-off):

- `stats.nba.com`'s **live** endpoints are known to be unreliable and
  prone to hanging/timing out unpredictably, independent of where the
  request comes from — see
  [swar/nba_api#633](https://github.com/swar/nba_api/issues/633)
  (Feb 2026: a *second* consecutive request in the same script times
  out, even run locally), and older recurring reports:
  [#176](https://github.com/swar/nba_api/issues/176),
  [#125](https://github.com/swar/nba_api/issues/125),
  [#320](https://github.com/swar/nba_api/issues/320).
- Cloud/datacenter IP blocking is a *real, separately documented*
  problem for this API (e.g.
  [#101](https://github.com/swar/nba_api/issues/101) on EC2,
  [this write-up](https://medium.com/@inman.justin/working-around-nba-coms-ip-ban-for-cloud-hosted-nba-api-apps-90326ab2632c)) —
  it just isn't what's causing *this specific* timeout, since a
  residential IP reproduces it identically.
- Sending the correct headers (already built into `nba_api`) does not
  fix it either, as tested earlier in this notebook.

**On VPN as a workaround** (commonly suggested in the `nba_api`
community for this exact symptom): the evidence is mixed, so this is
documented as a known option, not adopted for this project. Some users
route through a VPS/VPN IP specifically to escape *cloud*-IP blocking
(see the write-up above) — but that isn't the failure mode confirmed
here. Other reports go the opposite way:
[#30](https://github.com/swar/nba_api/issues/30) describes a VPN
*causing* the API to hang, fixed only by disconnecting it. Given a
residential IP already reproduces the timeout, there's no strong reason
to expect a VPN would reliably fix it for this project, so it is **not**
being built into `fetch.py` or the dev setup — noted here only as
something to try manually if `fetch.py` proves consistently unusable
otherwise.

### Design implications for `fetch.py` (task 1.3)

1. **Throttle every live call** with a small delay (`time.sleep`,
   ~0.6-1s) between requests, even on success — still correct, and
   `nba_api`'s own docs recommend it independent of this issue.
2. **Fail loud, not silent**: wrap each call with a bounded timeout and
   surface a clear error (not a hang) when a request stalls, so a
   pipeline run fails fast instead of hanging for minutes per endpoint.
   This matters *more* now that the failure is confirmed to be an
   `stats.nba.com` reliability issue rather than a one-off sandbox
   artifact — `fetch.py` should expect and handle it as routine, e.g.
   with a small number of retries before giving up on an endpoint.
3. **Cache raw responses to `data/raw/`** as soon as a call succeeds, so
   `clean.py` (task 1.4) can be developed and re-run against saved data
   without re-hitting a flaky API every time.
4. **No IP/network workaround** (proxy, VPN, alternate hosting) is being
   adopted — the timeouts are not IP-specific, so none of those would
   reliably fix it. Retrying and caching are the resilience strategy.


## Endpoints selected for V1

| Endpoint | Used by | Notes |
|---|---|---|
| `players` / `teams` (static) | id lookup for all pages | no network, always available |
| `CommonPlayerInfo` | player profile page (1.6) | bio data (position, height, draft, etc.) |
| `PlayerCareerStats` | player profile + comparison pages (1.6/1.7) | season-by-season stats |
| `LeagueGameLog` | not used in V1 | kept in mind for V2 (per-game granularity for ML features) |

Next: task 1.3 builds `fetch.py` around these endpoints and the
constraints documented above.
